In [1]:
import sys
sys.path.insert(0, "/Users/aidavillalba/Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy")
from multimodalpy import main
import os
import numpy as np
import geopandas as gpd
import seaborn as sns
import pandas as pd
from shapely import wkt
import shapely

from shapely.errors import ShapelyError
from dataclasses import dataclass, field

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Union

import numpy as np
import pandas as pd
import shapely

try:
    import geopandas as gpd
except ImportError:  # geopandas es opcional: solo hace falta para
    gpd = None        # leer GeoJSON / Shapefile / GeoPackage.

NAP_API_KEY= "44a2e8d8-fab3-4d87-940d-d7788a7dac94"

output_folder = "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia"
boundaries = "/Users/aidavillalba/Desktop/descargas_multimodalpy/input_shapes/capa_Valencia.shp"

# Functions

In [2]:



# ==========================================================================
# Esquemas / constantes
# ==========================================================================
NODE_COLS_OSM = ["node_id", "node_role", "geometry"]
EDGE_COLS_OSM = ["edge_id", "from_node_id", "to_node_id", "highway", "lanes",
                 "maxspeed", "name", "oneway", "reversed", "length", "tts",
                 "geometry"]

VALID_NODE_ROLES_OSM = {"intersection", "linear_vertex", "through_endpoint", "endpoint"}

# base OSM highway tags; compound values ("a;b") are checked part-by-part
KNOWN_HIGHWAY_TAGS = {
    "motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link", "cycleway", "footway",
    "secondary", "secondary_link", "tertiary", "tertiary_link", "unclassified",
    "residential", "living_street", "service", "busway", "pedestrian", "track",
    "road",
}

NODE_COLS_GTFS = ["node_id", "stop_name", "geometry"]
EDGE_COLS_GTFS = [
    "edge_id", "from_node_id", "to_node_id", "route_id", "route_short_name",
    "route_long_name", "trip_count", "tts_mean", "tts_min", "tts_max",
    "tts_mean_peak_am", "tts_mean_peak_pm", "tts_mean_rest_of_day",
    "trip_count_peak_am", "trip_count_peak_pm", "trip_count_rest_of_day",
    "tts_mean_weekday", "trip_count_weekday", "tts_mean_weekend",
    "trip_count_weekend", "days_of_week_summary", "hourly_travel_times",
    "hourly_trip_counts", "tts_mean_peak_am_weekday", "trip_count_peak_am_weekday",
    "tts_mean_peak_am_weekend", "trip_count_peak_am_weekend",
    "tts_mean_peak_pm_weekday", "trip_count_peak_pm_weekday",
    "tts_mean_peak_pm_weekend", "trip_count_peak_pm_weekend",
    "tts_mean_rest_of_day_weekday", "trip_count_rest_of_day_weekday",
    "tts_mean_rest_of_day_weekend", "trip_count_rest_of_day_weekend", "geometry",
]

VALID_DAYS = {"monday", "tuesday", "wednesday", "thursday", "friday",
              "saturday", "sunday"}

COORD_TOL = 0.01       # metros: extremos de geometria vs coords del nodo, en CRS proyectado
LENGTH_REL_TOL = 0.02  # 2% de tolerancia relativa entre la col `length` y la longitud geometrica

# Extensiones de archivo vectorial soportadas ademas de CSV/TSV
VECTOR_EXTENSIONS = {".geojson", ".json", ".shp", ".gpkg"}


# ==========================================================================
# I/O multi-formato: CSV (WKT) / GeoJSON / Shapefile / GeoPackage
# ==========================================================================
def load_table(path: Union[str, Path], layer: Optional[str] = None) -> pd.DataFrame:
    """
    Carga una tabla de nodos/aristas sin importar el formato de origen.

    - .csv / .tsv     -> lectura plana con pandas; la columna `geometry` (si
                          existe) queda como texto WKT y la parsea despues
                          `_parse_geometries`.
    - .geojson/.json  -> geopandas; la columna `geometry` queda como objetos
                          shapely ya parseados.
    - .shp            -> geopandas (requiere los .dbf/.shx/.prj junto al .shp).
                          OJO: ESRI Shapefile trunca nombres de columna a 10
                          caracteres, asi que columnas como "from_node_id"
                          pueden llegar renombradas/perdidas. `load_table` no
                          puede adivinar el nombre original de forma fiable
                          (varias columnas largas de este esquema truncan al
                          mismo prefijo de 10 caracteres), asi que si vas a
                          usar Shapefile como origen, renombra esas columnas
                          antes de pasar la tabla a los checks, o mejor usa
                          GeoJSON/GeoPackage, que no tienen ese limite.
    - .gpkg           -> geopandas; usa `layer=` para elegir la capa cuando el
                          GeoPackage contiene varias (p. ej. nodos y aristas
                          en el mismo .gpkg).

    Devuelve un DataFrame/GeoDataFrame con una columna `geometry`.
    """
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in {".csv", ".tsv"}:
        sep = "\t" if suffix == ".tsv" else ","
        return pd.read_csv(path, sep=sep)

    if suffix in VECTOR_EXTENSIONS:
        if gpd is None:
            raise ImportError(
                "geopandas es necesario para leer GeoJSON/Shapefile/GeoPackage. "
                "Instalalo con: pip install geopandas --break-system-packages"
            )
        read_kwargs = {"layer": layer} if (suffix == ".gpkg" and layer) else {}
        gdf = gpd.read_file(path, **read_kwargs)
        if gdf.geometry.name != "geometry":
            gdf = gdf.rename_geometry("geometry")
        return gdf

    raise ValueError(
        f"Extension no soportada '{suffix}' para {path}. "
        "Formatos validos: .csv, .tsv, .geojson, .json, .shp, .gpkg"
    )


def _ensure_table(obj: Union[str, Path, pd.DataFrame],
                   layer: Optional[str] = None) -> pd.DataFrame:
    """Acepta un DataFrame ya cargado, o una ruta que se carga con load_table."""
    if isinstance(obj, (str, Path)):
        return load_table(obj, layer=layer)
    return obj


# ==========================================================================
# Infraestructura de checks compartida (antes duplicada OSM/GTFS)
# ==========================================================================
@dataclass
class CheckResult:
    name: str
    level: str          # "fail" | "warn"
    n_issues: int
    issues: Optional[pd.DataFrame] = None
    note: str = ""


def _fmt(r: CheckResult) -> str:
    tag = "OK" if r.n_issues == 0 else f"{r.level.upper()} ({r.n_issues})"
    return f"[{tag}] {r.name}" + (f" — {r.note}" if r.note else "")


def _record(results, name, level, subset, note=""):
    r = CheckResult(name=name, level=level, n_issues=len(subset),
                     issues=subset if len(subset) else None, note=note)
    results.append(r)
    print(_fmt(r))
    if r.n_issues and r.issues is not None:
        # `subset` suele ser un DataFrame (tiene .head()), pero check_schema
        # pasa una lista simple de nombres de columna, que no lo tiene.
        if hasattr(r.issues, "head"):
            preview = r.issues.head(5)
            print(preview.to_string() if hasattr(preview, "to_string") else preview)
        else:
            print(list(r.issues)[:5])
        print()
    return r


def _col(df: pd.DataFrame, name: str) -> pd.Series:
    """
    Devuelve df[name] si existe; si no, una Serie de NaN del mismo largo.

    Esto evita que un `KeyError` corte toda la ejecucion cuando a la tabla de
    entrada real le falta una columna esperada (esquema distinto, columna
    truncada por Shapefile, etc.). El check de esquema (`check_schema`) ya
    reporta explicitamente que columna falta; con `_col`, los checks
    posteriores que la usan simplemente marcan todas las filas como
    problema (NaN no pasa "not null", no hace match en `.isin`, etc.) en vez
    de abortar el script entero.
    """
    if name in df.columns:
        return df[name]
    return pd.Series([np.nan] * len(df), index=df.index, name=name)


def _missing(df: pd.DataFrame, cols: list) -> list:
    """Lista de columnas de `cols` que NO estan en `df`, preservando orden."""
    return [c for c in cols if c not in df.columns]


def check_schema(df, expected_cols, name, results):
    print(f"--- {name}: schema ---")
    missing = set(expected_cols) - set(df.columns)
    extra = set(df.columns) - set(expected_cols)
    _record(results, f"{name}: expected columns present", "fail",
            sorted(missing))
    if extra:
        print(f"[INFO] {name}: extra/unexpected columns: {sorted(extra)}")
    print()


def _parse_geometries(geom_series: pd.Series) -> np.ndarray:
    """
    Devuelve un array numpy de geometrias shapely a partir de una columna que
    puede contener strings WKT (lectura plana de CSV) u objetos shapely ya
    parseados (p. ej. la columna geometry de un GeoDataFrame leido desde
    GeoJSON/Shapefile/GeoPackage). Las entradas mal formadas o vacias se
    convierten en None.
    """
    values = geom_series.to_numpy()
    is_str = np.array([isinstance(v, str) for v in values])

    out = np.empty(len(values), dtype=object)
    if is_str.any():
        str_vals = np.where(is_str, values, "")
        parsed = shapely.from_wkt(str_vals, on_invalid="ignore")
        out[is_str] = parsed[is_str]

    not_str = ~is_str
    if not_str.any():
        for i in np.where(not_str)[0]:
            v = values[i]
            out[i] = v if isinstance(v, shapely.Geometry) else None

    return out


def _multi_value_bad_mask(series: pd.Series, positive_only: bool) -> pd.Series:
    """
    Tags OSM como lanes/maxspeed pueden venir separados por ';' ("40;50", "3;2").
    Marca la fila como mala solo si no es nula y NO todas las partes separadas
    por ';' parsean como numeros (y, si positive_only, todas > 0).
    """
    def bad(val):
        if pd.isna(val):
            return False
        parts = str(val).split(";")
        nums = pd.to_numeric(parts, errors="coerce")
        if pd.isna(nums).any():
            return True
        if positive_only and (nums <= 0).any():
            return True
        return False
    return series.apply(bad)


def _parse_hourly_string(s) -> dict:
    """'00-01:120|01-02:120|...' -> {'00-01': 120.0, ...}. Dict vacio si NaN/mal formado."""
    if pd.isna(s) or not str(s):
        return {}
    out = {}
    try:
        for part in str(s).split("|"):
            hour, val = part.split(":")
            out[hour] = float(val)
    except (ValueError, IndexError):
        return {}
    return out


def _topology_mismatch_mask(edges: pd.DataFrame, edge_geoms: np.ndarray,
                             node_id_to_geom: dict, ok_geom: np.ndarray) -> np.ndarray:
    """
    Comun a OSM y GTFS: compara los extremos de cada LINESTRING de arista
    contra las coordenadas de sus nodos from/to (dentro de COORD_TOL).
    Las filas cuyo nodo from/to no existe en `node_id_to_geom` se omiten
    (ya quedan cubiertas por el check de integridad referencial).
    """
    mismatch_mask = np.zeros(len(edges), dtype=bool)
    from_ids = _col(edges, "from_node_id").to_numpy()
    to_ids = _col(edges, "to_node_id").to_numpy()
    for i in np.where(ok_geom)[0]:
        from_pt = node_id_to_geom.get(from_ids[i])
        to_pt = node_id_to_geom.get(to_ids[i])
        if from_pt is None or to_pt is None:
            continue
        coords = list(edge_geoms[i].coords)
        start, end = coords[0], coords[-1]
        d_start = ((start[0] - from_pt.x) ** 2 + (start[1] - from_pt.y) ** 2) ** 0.5
        d_end = ((end[0] - to_pt.x) ** 2 + (end[1] - to_pt.y) ** 2) ** 0.5
        if d_start > COORD_TOL or d_end > COORD_TOL:
            mismatch_mask[i] = True
    return mismatch_mask


def _summarize(results: list) -> None:
    n_fail = sum(1 for r in results if r.level == "fail" and r.n_issues)
    n_warn = sum(1 for r in results if r.level == "warn" and r.n_issues)
    n_ok = sum(1 for r in results if r.n_issues == 0)
    print("=== SUMMARY ===")
    print(f"{n_ok} passed, {n_fail} failed, {n_warn} warned "
          f"(out of {len(results)} checks)")


# ==========================================================================
# OSM checks
# ==========================================================================
def check_nodes_osm(nodes: pd.DataFrame, results: list) -> np.ndarray:
    print("=== NODES ===")
    check_schema(nodes, NODE_COLS_OSM, "nodes", results)

    _record(results, "no fully duplicate rows", "fail", nodes[nodes.duplicated()])
    node_id = _col(nodes, "node_id")
    _record(results, "node_id is unique", "fail",
            nodes[node_id.duplicated(keep=False)].sort_values("node_id") if "node_id" in nodes.columns
            else nodes.iloc[0:0])
    _record(results, "node_id not null", "fail", nodes[node_id.isna()])
    geometry = _col(nodes, "geometry")
    _record(results, "geometry not null", "fail", nodes[geometry.isna()])

    if "node_role" in nodes.columns:
        bad_roles = nodes[~nodes["node_role"].isin(VALID_NODE_ROLES_OSM)]
    else:
        bad_roles = nodes  # columna ausente: no se puede validar ninguna fila -> todas fallan
    _record(results, f"node_role in {sorted(VALID_NODE_ROLES_OSM)}", "fail", bad_roles)

    geoms = _parse_geometries(geometry)
    is_bad = np.array([g is None or g.geom_type != "Point" or not g.is_valid
                        for g in geoms])
    _record(results, "geometry parses as valid POINT", "fail", nodes[is_bad])
    print()
    return geoms


def check_edges_osm(edges: pd.DataFrame, nodes: pd.DataFrame, node_geoms: np.ndarray,
                     results: list) -> np.ndarray:
    print("=== EDGES ===")
    check_schema(edges, EDGE_COLS_OSM, "edges", results)

    _record(results, "no fully duplicate rows", "fail", edges[edges.duplicated()])
    _record(results, "edge_id not null", "fail", edges[_col(edges, "edge_id").isna()])
    from_node_id = _col(edges, "from_node_id")
    to_node_id = _col(edges, "to_node_id")
    _record(results, "from_node_id not null", "fail", edges[from_node_id.isna()])
    _record(results, "to_node_id not null", "fail", edges[to_node_id.isna()])

    node_id_to_geom = dict(zip(_col(nodes, "node_id"), node_geoms))
    node_ids = set(node_id_to_geom)
    _record(results, "from_node_id exists in nodes table", "fail",
            edges[~from_node_id.isin(node_ids)])
    _record(results, "to_node_id exists in nodes table", "fail",
            edges[~to_node_id.isin(node_ids)])

    _record(results, "no self-loop edges (from == to)", "fail",
            edges[from_node_id == to_node_id])

    dup_key_cols = _missing(edges, ["edge_id", "from_node_id", "to_node_id"])
    if dup_key_cols:
        print(f"[SKIP] no duplicate (edge_id, from_node_id, to_node_id) rows — "
              f"falta(n) columna(s): {dup_key_cols}")
    else:
        _record(results, "no duplicate (edge_id, from_node_id, to_node_id) rows", "warn",
                edges[edges.duplicated(subset=["edge_id", "from_node_id", "to_node_id"],
                                        keep=False)],
                note="often expected for bidirectional OSM ways sharing one edge_id")

    length = _col(edges, "length")
    tts = _col(edges, "tts")
    _record(results, "length not null", "fail", edges[length.isna()])
    _record(results, "length is non-negative", "fail", edges[length < 0])
    _record(results, "tts (travel time) is non-negative", "fail", edges[tts < 0])

    oneway = _col(edges, "oneway")
    reversed_col = _col(edges, "reversed")
    _record(results, "oneway is boolean dtype", "fail",
            edges if oneway.dtype != bool else edges.iloc[0:0])
    _record(results, "reversed is boolean dtype", "fail",
            edges if reversed_col.dtype != bool else edges.iloc[0:0])

    geometry = _col(edges, "geometry")
    edge_geoms = _parse_geometries(geometry)
    is_bad_line = np.array([g is None or g.geom_type != "LineString" or not g.is_valid
                             for g in edge_geoms])
    _record(results, "geometry parses as valid LINESTRING", "fail", edges[is_bad_line])

    ok_geom = ~is_bad_line
    mismatch_mask = _topology_mismatch_mask(edges, edge_geoms, node_id_to_geom, ok_geom)
    _record(results, "edge geometry endpoints match node coords", "fail",
            edges[mismatch_mask], note=f"tolerance {COORD_TOL} m")

    geom_len = np.array([g.length if g is not None else np.nan for g in edge_geoms])
    with np.errstate(invalid="ignore", divide="ignore"):
        rel_diff = np.abs(geom_len - length.to_numpy()) / geom_len
    len_mismatch = ok_geom & (rel_diff > LENGTH_REL_TOL)
    _record(results, "`length` column matches geometry length", "warn",
            edges[len_mismatch], note=f"tolerance {LENGTH_REL_TOL:.0%}")

    bad_speed = edges[_multi_value_bad_mask(_col(edges, "maxspeed"), positive_only=True)]
    _record(results, "maxspeed numeric (incl. ';'-separated values) when present",
            "fail", bad_speed)

    bad_lanes = edges[_multi_value_bad_mask(_col(edges, "lanes"), positive_only=True)]
    _record(results, "lanes numeric & positive (incl. ';'-separated values) when present",
            "fail", bad_lanes)

    def unknown_highway(val):
        if pd.isna(val):
            return True
        return not set(str(val).split(";")).issubset(KNOWN_HIGHWAY_TAGS)
    bad_hwy = edges[_col(edges, "highway").apply(unknown_highway)]
    _record(results, "highway tag(s) within known OSM vocabulary", "warn", bad_hwy)
    print()
    return edge_geoms


def check_graph_consistency_osm(nodes: pd.DataFrame, edges: pd.DataFrame, results: list):
    print("=== CROSS-TABLE CONSISTENCY ===")
    needed = _missing(edges, ["from_node_id", "to_node_id"]) + _missing(nodes, ["node_id", "node_role"])
    if needed:
        print(f"[SKIP] cross-table consistency — falta(n) columna(s): {sorted(set(needed))}")
        print()
        return

    used_ids = set(edges["from_node_id"]) | set(edges["to_node_id"])
    orphan_nodes = nodes[~nodes["node_id"].isin(used_ids)]
    print(f"[INFO] nodes not referenced by any edge: {len(orphan_nodes)}")

    # Grado topologico: cuenta conexiones DISTINTAS por nodo, no filas crudas.
    # Varias aristas pueden compartir el mismo par (from_node_id, to_node_id)
    # (p. ej. ways bidireccionales de OSM, o el mismo segmento fisico con mas
    # de un edge_id/ruta) y deben contar como UNA sola conexion.
    node_pairs = edges[["from_node_id", "to_node_id"]].to_numpy()
    undirected_pairs = {tuple(sorted(p)) for p in node_pairs}
    unique_edges = pd.DataFrame(undirected_pairs, columns=["a", "b"])
    degree = pd.concat([unique_edges["a"], unique_edges["b"]]).value_counts()

    endpoint_nodes = nodes.loc[nodes["node_role"] == "endpoint"].copy()
    endpoint_nodes["degree"] = endpoint_nodes["node_id"].map(degree).fillna(0).astype(int)
    hi_degree_endpoints = endpoint_nodes[endpoint_nodes["degree"] > 2].sort_values(
        "degree", ascending=False
    )
    print(f"[INFO] 'endpoint' nodes with degree > 2 (unexpected for a dead-end role, "
          f"degree = count of distinct connected nodes): {len(hi_degree_endpoints)}")

    if len(hi_degree_endpoints):
        print(hi_degree_endpoints[["node_id", "node_role", "degree", "geometry"]]
              .to_string(index=False))
    print()


def run_all_checks_osm(nodes: Union[str, Path, pd.DataFrame],
                        edges: Union[str, Path, pd.DataFrame],
                        nodes_layer: Optional[str] = None,
                        edges_layer: Optional[str] = None):
    """
    `nodes`/`edges` pueden ser DataFrames ya cargados, o rutas a archivos
    .csv/.tsv/.geojson/.shp/.gpkg — se cargan automaticamente con
    `load_table`. `nodes_layer`/`edges_layer` solo aplican a GeoPackage.
    """
    nodes = _ensure_table(nodes, layer=nodes_layer)
    edges = _ensure_table(edges, layer=edges_layer)

    results: list[CheckResult] = []
    node_geoms = check_nodes_osm(nodes, results)
    check_edges_osm(edges, nodes, node_geoms, results)
    check_graph_consistency_osm(nodes, edges, results)
    _summarize(results)
    return results


# ==========================================================================
# GTFS checks
# ==========================================================================
def check_nodes_gtfs(nodes: pd.DataFrame, results: list) -> np.ndarray:
    print("=== NODES (stops) ===")
    check_schema(nodes, NODE_COLS_GTFS, "nodes", results)

    _record(results, "no fully duplicate rows", "fail", nodes[nodes.duplicated()])
    node_id = _col(nodes, "node_id")
    _record(results, "node_id is unique", "fail",
            nodes[node_id.duplicated(keep=False)].sort_values("node_id") if "node_id" in nodes.columns
            else nodes.iloc[0:0])
    _record(results, "node_id not null", "fail", nodes[node_id.isna()])
    _record(results, "stop_name not null", "warn", nodes[_col(nodes, "stop_name").isna()])
    geometry = _col(nodes, "geometry")
    _record(results, "geometry not null", "fail", nodes[geometry.isna()])

    geoms = _parse_geometries(geometry)
    is_bad = np.array([g is None or g.geom_type != "Point" or not g.is_valid
                        for g in geoms])
    _record(results, "geometry parses as valid POINT", "fail", nodes[is_bad])
    print()
    return geoms


def check_edges_gtfs(edges: pd.DataFrame, nodes: pd.DataFrame, node_geoms: np.ndarray,
                      results: list) -> np.ndarray:
    print("=== EDGES (route segments) ===")
    check_schema(edges, EDGE_COLS_GTFS, "edges", results)

    _record(results, "edge_id not null", "fail", edges[_col(edges, "edge_id").isna()])
    from_node_id = _col(edges, "from_node_id")
    to_node_id = _col(edges, "to_node_id")
    _record(results, "from_node_id not null", "fail", edges[from_node_id.isna()])
    _record(results, "to_node_id not null", "fail", edges[to_node_id.isna()])
    _record(results, "route_id not null", "fail", edges[_col(edges, "route_id").isna()])

    # `nodes` puede ser un subconjunto espacial de paradas que no incluya
    # todas las que atraviesa una ruta que pasa por la zona -> warning, no fail.
    node_id_to_geom = dict(zip(_col(nodes, "node_id"), node_geoms))
    node_ids = set(node_id_to_geom)
    _record(results, "from_node_id exists in nodes table", "warn",
            edges[~from_node_id.isin(node_ids)],
            note="expected if nodes.csv is a spatial subset of stops")
    _record(results, "to_node_id exists in nodes table", "warn",
            edges[~to_node_id.isin(node_ids)],
            note="expected if nodes.csv is a spatial subset of stops")

    _record(results, "no self-loop edges (from == to)", "fail",
            edges[from_node_id == to_node_id])

    geometry = _col(edges, "geometry")
    edge_geoms = _parse_geometries(geometry)
    is_bad_line = np.array([g is None or g.geom_type != "LineString" or not g.is_valid
                             for g in edge_geoms])
    _record(results, "geometry parses as valid LINESTRING", "fail", edges[is_bad_line])

    ok_geom = ~is_bad_line
    mismatch_mask = _topology_mismatch_mask(edges, edge_geoms, node_id_to_geom, ok_geom)
    _record(results, "edge geometry endpoints match node coords", "fail",
            edges[mismatch_mask], note=f"tolerance {COORD_TOL} m, only checked "
                                        "where both endpoints exist in nodes")

    trip_count = _col(edges, "trip_count")
    tts_mean = _col(edges, "tts_mean")
    tts_min = _col(edges, "tts_min")
    tts_max = _col(edges, "tts_max")
    _record(results, "trip_count is positive", "fail", edges[trip_count <= 0])
    _record(results, "tts_mean is non-negative", "fail", edges[tts_mean < 0])
    _record(results, "tts_min <= tts_mean <= tts_max", "fail",
            edges[~((tts_min <= tts_mean) & (tts_mean <= tts_max))])

    breakdown_cols = ["trip_count_peak_am", "trip_count_peak_pm", "trip_count_rest_of_day"]
    missing_breakdown = _missing(edges, breakdown_cols)
    if missing_breakdown:
        print(f"[SKIP] trip_count == peak_am + peak_pm + rest_of_day breakdown — "
              f"falta(n) columna(s): {missing_breakdown}")
    else:
        parts = edges[breakdown_cols].fillna(0)
        calc = parts.sum(axis=1)
        _record(results, "trip_count == peak_am + peak_pm + rest_of_day breakdown",
                "fail", edges[calc != trip_count])

    hourly_trip_counts = _col(edges, "hourly_trip_counts")
    hourly_sum = hourly_trip_counts.apply(
        lambda s: sum(_parse_hourly_string(s).values()))
    _record(results, "trip_count == sum(hourly_trip_counts)", "fail",
            edges[hourly_sum != trip_count])

    hourly_travel_times = _col(edges, "hourly_travel_times")

    def bad_hourly_pair(i):
        tt = _parse_hourly_string(hourly_travel_times.iloc[i])
        tc = _parse_hourly_string(hourly_trip_counts.iloc[i])
        return (not tt) or (not tc) or (set(tt) != set(tc))
    bad_hourly_mask = pd.Series([bad_hourly_pair(i) for i in range(len(edges))], index=edges.index)
    _record(results, "hourly_travel_times / hourly_trip_counts parse & align",
            "fail", edges[bad_hourly_mask])

    def bad_days(val):
        if pd.isna(val):
            return True
        return not set(str(val).split("|")).issubset(VALID_DAYS)
    _record(results, "days_of_week_summary uses valid day names", "fail",
            edges[_col(edges, "days_of_week_summary").apply(bad_days)])

    print()
    return edge_geoms


def check_route_consistency_gtfs(edges: pd.DataFrame, results: list):
    print("=== ROUTE-LEVEL CONSISTENCY ===")
    needed = _missing(edges, ["route_id", "route_short_name", "route_long_name"])
    if needed:
        print(f"[SKIP] route-level consistency — falta(n) columna(s): {needed}")
        print()
        return

    grp = edges.groupby("route_id")[["route_short_name", "route_long_name"]].nunique()
    inconsistent_routes = grp[(grp["route_short_name"] > 1) | (grp["route_long_name"] > 1)]
    print(f"[INFO] route_id values with inconsistent short/long name across rows: "
          f"{len(inconsistent_routes)}")
    if len(inconsistent_routes):
        print(inconsistent_routes.to_string())
    print()

    seg_counts = edges.groupby("route_id").size()
    print(f"[INFO] route_id count: {edges['route_id'].nunique()}; "
          f"segments per route — min {seg_counts.min()}, "
          f"median {int(seg_counts.median())}, max {seg_counts.max()}")
    print()


def run_all_checks_gtfs(nodes: Union[str, Path, pd.DataFrame],
                         edges: Union[str, Path, pd.DataFrame],
                         nodes_layer: Optional[str] = None,
                         edges_layer: Optional[str] = None):
    """
    `nodes`/`edges` pueden ser DataFrames ya cargados, o rutas a archivos
    .csv/.tsv/.geojson/.shp/.gpkg — se cargan automaticamente con
    `load_table`. `nodes_layer`/`edges_layer` solo aplican a GeoPackage.
    """
    nodes = _ensure_table(nodes, layer=nodes_layer)
    edges = _ensure_table(edges, layer=edges_layer)

    results: list[CheckResult] = []
    node_geoms = check_nodes_gtfs(nodes, results)
    check_edges_gtfs(edges, nodes, node_geoms, results)
    check_route_consistency_gtfs(edges, results)
    _summarize(results)
    return results

# GeoJSON

In [3]:
main(
    area_name="Valencia",
    modes=["train", "bus", "driving", "bike", "walking"],
    output_path=output_folder,
    crs="epsg:25830",
    output_file_type="geojson",
    boundaries_path= boundaries,
    api_key=NAP_API_KEY
)

[aviso] No se pudo normalizar el feed 20260727_080002_BUS_Samar.zip: 'stop_lat'
[aviso] No se pudo normalizar el feed 20260908_000017_BUS_Cantabria.zip: Python int too large to convert to C long
[aviso] No se pudo normalizar el feed 20260908_120014_RENFE_CERCA.zip: 'stop_sequence'
{
  "area_name": "Valencia",
  "modes": [
    "train",
    "bus",
    "driving",
    "bike",
    "walking"
  ],
  "crs": "epsg:25830",
  "output_file_type": "geojson",
  "output_path": "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia",
  "osm_modes": [
    "driving",
    "bike",
    "walking"
  ],
  "gtfs_datasets": [
    "20260908_000019_bus_alsa",
    "20260908_020020_genvalenciana_interurbano",
    "20260901_200004_emt_valencia",
    "20260902_080005_metro_valencia",
    "20260702_060101_ouigo",
    "20260908_000005_renfe_avld"
  ],
  "files": [
    "study_area_boundary.geojson",
    "osm_driving_nodes.geojson",
    "osm_driving_edges.geojson",
    "osm_bike_nodes.geojson",
    "osm_bike

{'area_name': 'Valencia',
 'modes': ['train', 'bus', 'driving', 'bike', 'walking'],
 'crs': 'epsg:25830',
 'output_file_type': 'geojson',
 'output_path': '/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia',
 'osm_modes': ['driving', 'bike', 'walking'],
 'gtfs_datasets': ['20260908_000019_bus_alsa',
  '20260908_020020_genvalenciana_interurbano',
  '20260901_200004_emt_valencia',
  '20260902_080005_metro_valencia',
  '20260702_060101_ouigo',
  '20260908_000005_renfe_avld'],
 'files': ['study_area_boundary.geojson',
  'osm_driving_nodes.geojson',
  'osm_driving_edges.geojson',
  'osm_bike_nodes.geojson',
  'osm_bike_edges.geojson',
  'osm_walking_nodes.geojson',
  'osm_walking_edges.geojson',
  'gtfs_20260908_000019_bus_alsa_nodes.geojson',
  'gtfs_20260908_000019_bus_alsa_edges.geojson',
  'gtfs_20260908_020020_genvalenciana_interurbano_nodes.geojson',
  'gtfs_20260908_020020_genvalenciana_interurbano_edges.geojson',
  'gtfs_20260901_200004_emt_valencia_nodes.geojson',
 

In [4]:
#Driving
df_driving_nodes = gpd.read_file(f"{output_folder}/osm_driving_nodes.geojson")
df_driving_edges = gpd.read_file(f"{output_folder}/osm_driving_edges.geojson")

#Walking
df_walking_nodes = gpd.read_file(f"{output_folder}/osm_walking_nodes.geojson")
df_walking_edges = gpd.read_file(f"{output_folder}/osm_walking_edges.geojson")

#Bike
df_bike_nodes = gpd.read_file(f"{output_folder}/osm_bike_nodes.geojson")
df_bike_edges = gpd.read_file(f"{output_folder}/osm_bike_edges.geojson")

#Train
df_train_nodes = gpd.read_file(f"{output_folder}/gtfs_20260902_080005_metro_valencia_nodes.geojson")
df_train_edges = gpd.read_file(f"{output_folder}/gtfs_20260902_080005_metro_valencia_edges.geojson")

#Bus
df_bus_nodes = gpd.read_file(f"{output_folder}/gtfs_20260908_020020_genvalenciana_interurbano_nodes.geojson")
df_bus_edges = gpd.read_file(f"{output_folder}/gtfs_20260908_020020_genvalenciana_interurbano_edges.geojson")

### Driving

In [5]:
run_all_checks_osm(df_driving_nodes, df_driving_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[OK] edges: expected columns present

[OK] no fully duplicate rows
[OK] edge_id not null
[OK] from_node_id not null
[OK] to_node_id not null
[OK] from_node_id exists in nodes table
[OK] to_node_id exists in nodes table
[OK] no self-loop edges (from == to)
[OK] no duplicate (edge_id, from_node_id, to_node_id) rows — often expected for bidirectional OSM ways sharing one edge_id
[OK] length not null
[OK] length is non-negative
[OK] tts (travel time) is non-negative
[OK] oneway is boolean dtype
[OK] reversed is boolean dtype
[OK] geometry parses as valid LINESTRING
[OK] edge geometry endpoints match node coords — tolerance 0.01 m
[OK] `length` column matches geometry 

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry parses as valid POINT', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edges: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edge_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckRe

### Walking

In [6]:
run_all_checks_osm(df_walking_nodes, df_walking_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[OK] edges: expected columns present

[OK] no fully duplicate rows
[OK] edge_id not null
[OK] from_node_id not null
[OK] to_node_id not null
[OK] from_node_id exists in nodes table
[OK] to_node_id exists in nodes table
[OK] no self-loop edges (from == to)
[OK] no duplicate (edge_id, from_node_id, to_node_id) rows — often expected for bidirectional OSM ways sharing one edge_id
[OK] length not null
[OK] length is non-negative
[OK] tts (travel time) is non-negative
[OK] oneway is boolean dtype
[OK] reversed is boolean dtype
[OK] geometry parses as valid LINESTRING
[OK] edge geometry endpoints match node coords — tolerance 0.01 m
[OK] `length` column matches geometry 

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry parses as valid POINT', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edges: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edge_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckRe

### Bike

In [7]:
run_all_checks_osm(df_bike_nodes, df_bike_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[OK] edges: expected columns present

[OK] no fully duplicate rows
[OK] edge_id not null
[OK] from_node_id not null
[OK] to_node_id not null
[OK] from_node_id exists in nodes table
[OK] to_node_id exists in nodes table
[OK] no self-loop edges (from == to)
[OK] no duplicate (edge_id, from_node_id, to_node_id) rows — often expected for bidirectional OSM ways sharing one edge_id
[OK] length not null
[OK] length is non-negative
[OK] tts (travel time) is non-negative
[OK] oneway is boolean dtype
[OK] reversed is boolean dtype
[OK] geometry parses as valid LINESTRING
[OK] edge geometry endpoints match node coords — tolerance 0.01 m
[OK] `length` column matches geometry 

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry parses as valid POINT', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edges: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edge_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckRe

### Bus 

In [8]:
run_all_checks_osm(df_bus_nodes, df_bus_edges)

=== NODES ===
--- nodes: schema ---
[FAIL (1)] nodes: expected columns present
['node_role']

[INFO] nodes: extra/unexpected columns: ['stop_name']

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[FAIL (244)] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
        node_id                                      stop_name                        geometry
0    5252010122                     C. Sant Vicent Màrtir, 317  POINT (724939.125 4369674.708)
1    5252050117  Av. Fernando Abril Martorell - Hospital La Fe  POINT (725780.904 4369435.429)
2    5253010216                   Av. Amado Granell Mesado, 49  POINT (726843.164 4370824.708)
3    5904210110                              GV. Germanies, 49  POINT (725613.409 4371455.285)
4  925218000218                         Av. Real de Madrid, 15  POINT (724560.818 4368294.497)

[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[FAIL (8)] edges: 

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=1, issues=['node_role'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=244, issues=          node_id                                          stop_name  \
 0      5252010122                         C. Sant Vicent Màrtir, 317   
 1      5252050117      Av. Fernando Abril Martorell - Hospital La Fe   
 2      5253010216                       Av. Amado Granell Mesado, 49   
 3      5904210110                                  GV. Germanies, 49   
 4    925218000218               

### Train

In [9]:
run_all_checks_osm(df_train_nodes, df_train_edges)

=== NODES ===
--- nodes: schema ---
[FAIL (1)] nodes: expected columns present
['node_role']

[INFO] nodes: extra/unexpected columns: ['stop_name']

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[FAIL (64)] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
  node_id      stop_name                        geometry
0      20   València Sud  POINT (722833.496 4368895.713)
1      28      Beniferri  POINT (723646.019 4374498.324)
2      27       Campanar  POINT (724036.479 4373798.483)
3      26          Túria    POINT (724384.5 4373168.203)
4      25  Àngel Guimerà   POINT (724942.812 4372232.97)

[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[FAIL (8)] edges: expected columns present
['highway', 'lanes', 'length', 'maxspeed', 'name']

[INFO] edges: extra/unexpected columns: ['days_of_week_summary', 'hourly_travel_times', 'hourly_trip_counts', 'route_id', 'route_long_name', 'route_shor

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=1, issues=['node_role'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=64, issues=   node_id                          stop_name                        geometry
 0       20                       València Sud  POINT (722833.496 4368895.713)
 1       28                          Beniferri  POINT (723646.019 4374498.324)
 2       27                           Campanar  POINT (724036.479 4373798.483)
 3       26                              Túria    POINT (724384.5 4373168.203)
 4  

# Shapefile

In [10]:
main(
    area_name="Valencia",
    modes=["train", "bus", "driving", "bike", "walking"],
    output_path=output_folder,
    crs="epsg:25830",
    output_file_type="shapefile",
    boundaries_path= boundaries,
    api_key=NAP_API_KEY
)

[aviso] No se pudo normalizar el feed 20260727_080002_BUS_Samar.zip: 'stop_lat'
[aviso] No se pudo normalizar el feed 20260908_000017_BUS_Cantabria.zip: Python int too large to convert to C long
[aviso] No se pudo normalizar el feed 20260908_120014_RENFE_CERCA.zip: 'stop_sequence'


/Users/aidavillalba/Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy/multimodalpy/get_network.py:194: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(path, driver="ESRI Shapefile")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'matched_column' to 'matched_co'
  ogr_write(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'matched_value' to 'matched_va'
  ogr_write(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'matched_rows' to 'matched_ro'
  ogr_write(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'source_path

{
  "area_name": "Valencia",
  "modes": [
    "train",
    "bus",
    "driving",
    "bike",
    "walking"
  ],
  "crs": "epsg:25830",
  "output_file_type": "shapefile",
  "output_path": "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia",
  "osm_modes": [
    "driving",
    "bike",
    "walking"
  ],
  "gtfs_datasets": [
    "20260908_000019_bus_alsa",
    "20260908_020020_genvalenciana_interurbano",
    "20260901_200004_emt_valencia",
    "20260902_080005_metro_valencia",
    "20260702_060101_ouigo",
    "20260908_000005_renfe_avld"
  ],
  "files": [
    "study_area_boundary.shp",
    "osm_driving_nodes.shp",
    "osm_driving_edges.shp",
    "osm_bike_nodes.shp",
    "osm_bike_edges.shp",
    "osm_walking_nodes.shp",
    "osm_walking_edges.shp",
    "gtfs_20260908_000019_bus_alsa_nodes.shp",
    "gtfs_20260908_000019_bus_alsa_edges.shp",
    "gtfs_20260908_020020_genvalenciana_interurbano_nodes.shp",
    "gtfs_20260908_020020_genvalenciana_interurbano_edges.shp",
   

{'area_name': 'Valencia',
 'modes': ['train', 'bus', 'driving', 'bike', 'walking'],
 'crs': 'epsg:25830',
 'output_file_type': 'shapefile',
 'output_path': '/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia',
 'osm_modes': ['driving', 'bike', 'walking'],
 'gtfs_datasets': ['20260908_000019_bus_alsa',
  '20260908_020020_genvalenciana_interurbano',
  '20260901_200004_emt_valencia',
  '20260902_080005_metro_valencia',
  '20260702_060101_ouigo',
  '20260908_000005_renfe_avld'],
 'files': ['study_area_boundary.shp',
  'osm_driving_nodes.shp',
  'osm_driving_edges.shp',
  'osm_bike_nodes.shp',
  'osm_bike_edges.shp',
  'osm_walking_nodes.shp',
  'osm_walking_edges.shp',
  'gtfs_20260908_000019_bus_alsa_nodes.shp',
  'gtfs_20260908_000019_bus_alsa_edges.shp',
  'gtfs_20260908_020020_genvalenciana_interurbano_nodes.shp',
  'gtfs_20260908_020020_genvalenciana_interurbano_edges.shp',
  'gtfs_20260901_200004_emt_valencia_nodes.shp',
  'gtfs_20260901_200004_emt_valencia_edges.shp'

In [11]:
#Driving
df_driving_nodes = gpd.read_file(f"{output_folder}/osm_driving_nodes.shp")
df_driving_edges = gpd.read_file(f"{output_folder}/osm_driving_edges.shp")

#Walking
df_walking_nodes = gpd.read_file(f"{output_folder}/osm_walking_nodes.shp")
df_walking_edges = gpd.read_file(f"{output_folder}/osm_walking_edges.shp")

#Bike
df_bike_nodes = gpd.read_file(f"{output_folder}/osm_bike_nodes.shp")
df_bike_edges = gpd.read_file(f"{output_folder}/osm_bike_edges.shp")

#Train
df_train_nodes = gpd.read_file(f"{output_folder}/gtfs_20260902_080005_metro_valencia_nodes.shp")
df_train_edges = gpd.read_file(f"{output_folder}/gtfs_20260902_080005_metro_valencia_edges.shp")

#Bus
df_bus_nodes = gpd.read_file(f"{output_folder}/gtfs_20260908_020020_genvalenciana_interurbano_nodes.shp")
df_bus_edges = gpd.read_file(f"{output_folder}/gtfs_20260908_020020_genvalenciana_interurbano_edges.shp")

### Driving

In [12]:
run_all_checks_osm(df_driving_nodes, df_driving_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[FAIL (1)] edges: expected columns present
['from_node_id']

[INFO] edges: extra/unexpected columns: ['from_node_']

[OK] no fully duplicate rows
[OK] edge_id not null
[FAIL (77658)] from_node_id not null
    edge_id                     from_node_                     to_node_id      highway lanes maxspeed                   name  oneway  reversed     length       tts                                                     geometry
0   5011435                       25767293  driving_topology_node_0000002    secondary     5       50       Carrer de Xàtiva    True     False  13.335806  0.960178   LINESTRING (725813.199 4371905.836, 725799.924 4371904.26)
1   5011435  driv

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry parses as valid POINT', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edges: expected columns present', level='fail', n_issues=1, issues=['from_node_id'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edge_id not null', level='fail', n_issues=0, issues=None, note='

### Walking

In [13]:
run_all_checks_osm(df_walking_nodes, df_walking_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[FAIL (1)] edges: expected columns present
['from_node_id']

[INFO] edges: extra/unexpected columns: ['from_node_']

[OK] no fully duplicate rows
[OK] edge_id not null
[FAIL (387360)] from_node_id not null
     edge_id                     from_node_                     to_node_id      highway lanes maxspeed                   name  oneway  reversed    length       tts                                                     geometry
0  803049226                       25767290                    11420770018      footway  None     None                   None   False     False  8.794706  6.332188  LINESTRING (725727.356 4372260.811, 725723.068 4372268.485)
1  803049226    

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry parses as valid POINT', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edges: expected columns present', level='fail', n_issues=1, issues=['from_node_id'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edge_id not null', level='fail', n_issues=0, issues=None, note='

### Bike

In [14]:
run_all_checks_osm(df_bike_nodes, df_bike_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[FAIL (1)] edges: expected columns present
['from_node_id']

[INFO] edges: extra/unexpected columns: ['from_node_']

[OK] no fully duplicate rows
[OK] edge_id not null
[FAIL (201023)] from_node_id not null
     edge_id from_node_                  to_node_id     highway lanes maxspeed                               name  oneway  reversed      length        tts                                                     geometry
0    5098789   25767292                  6766690060  pedestrian  None     None         Carrer de Martínez Cubells   False     False   56.652682  16.596644  LINESTRING (725789.942 4372033.731, 725846.733 4372032.464)
1  583027084   25767292           

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry parses as valid POINT', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edges: expected columns present', level='fail', n_issues=1, issues=['from_node_id'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edge_id not null', level='fail', n_issues=0, issues=None, note='

### Train

In [15]:
run_all_checks_osm(df_train_nodes, df_train_edges)

=== NODES ===
--- nodes: schema ---
[FAIL (1)] nodes: expected columns present
['node_role']

[INFO] nodes: extra/unexpected columns: ['stop_name']

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[FAIL (64)] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
  node_id      stop_name                        geometry
0      20   València Sud  POINT (722833.496 4368895.713)
1      28      Beniferri  POINT (723646.019 4374498.324)
2      27       Campanar  POINT (724036.479 4373798.483)
3      26          Túria    POINT (724384.5 4373168.203)
4      25  Àngel Guimerà   POINT (724942.812 4372232.97)

[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[FAIL (9)] edges: expected columns present
['from_node_id', 'highway', 'lanes', 'length', 'maxspeed']

[INFO] edges: extra/unexpected columns: ['days_of_we', 'from_node_', 'hourly_tra', 'hourly_tri', 'route_id', 'route_long', 'route_shor', 'trip_c

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=1, issues=['node_role'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=64, issues=   node_id                          stop_name                        geometry
 0       20                       València Sud  POINT (722833.496 4368895.713)
 1       28                          Beniferri  POINT (723646.019 4374498.324)
 2       27                           Campanar  POINT (724036.479 4373798.483)
 3       26                              Túria    POINT (724384.5 4373168.203)
 4  

### Bus

In [16]:
run_all_checks_osm(df_bus_nodes, df_bus_edges)

=== NODES ===
--- nodes: schema ---
[FAIL (1)] nodes: expected columns present
['node_role']

[INFO] nodes: extra/unexpected columns: ['stop_name']

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[FAIL (244)] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
        node_id                                      stop_name                        geometry
0    5252010122                     C. Sant Vicent Màrtir, 317  POINT (724939.125 4369674.708)
1    5252050117  Av. Fernando Abril Martorell - Hospital La Fe  POINT (725780.904 4369435.429)
2    5253010216                   Av. Amado Granell Mesado, 49  POINT (726843.164 4370824.708)
3    5904210110                              GV. Germanies, 49  POINT (725613.409 4371455.285)
4  925218000218                         Av. Real de Madrid, 15  POINT (724560.818 4368294.497)

[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[FAIL (9)] edges: 

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=1, issues=['node_role'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=244, issues=          node_id                                          stop_name  \
 0      5252010122                         C. Sant Vicent Màrtir, 317   
 1      5252050117      Av. Fernando Abril Martorell - Hospital La Fe   
 2      5253010216                       Av. Amado Granell Mesado, 49   
 3      5904210110                                  GV. Germanies, 49   
 4    925218000218               

# GeoPackage

In [17]:
main(
    area_name="Valencia",
    modes=["train", "bus", "driving", "bike", "walking"],
    output_path=output_folder,
    crs="epsg:25830",
    output_file_type="geopackage",
    boundaries_path= boundaries,
    api_key=NAP_API_KEY
)

[aviso] No se pudo normalizar el feed 20260727_080002_BUS_Samar.zip: 'stop_lat'
[aviso] No se pudo normalizar el feed 20260908_000017_BUS_Cantabria.zip: Python int too large to convert to C long
[aviso] No se pudo normalizar el feed 20260908_120014_RENFE_CERCA.zip: 'stop_sequence'
{
  "area_name": "Valencia",
  "modes": [
    "train",
    "bus",
    "driving",
    "bike",
    "walking"
  ],
  "crs": "epsg:25830",
  "output_file_type": "geopackage",
  "output_path": "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia",
  "osm_modes": [
    "driving",
    "bike",
    "walking"
  ],
  "gtfs_datasets": [
    "20260908_000019_bus_alsa",
    "20260908_020020_genvalenciana_interurbano",
    "20260901_200004_emt_valencia",
    "20260902_080005_metro_valencia",
    "20260702_060101_ouigo",
    "20260908_000005_renfe_avld"
  ],
  "files": [
    "valencia.gpkg::study_area_boundary",
    "valencia.gpkg::osm_driving_nodes",
    "valencia.gpkg::osm_driving_edges",
    "valencia.gpkg:

{'area_name': 'Valencia',
 'modes': ['train', 'bus', 'driving', 'bike', 'walking'],
 'crs': 'epsg:25830',
 'output_file_type': 'geopackage',
 'output_path': '/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia',
 'osm_modes': ['driving', 'bike', 'walking'],
 'gtfs_datasets': ['20260908_000019_bus_alsa',
  '20260908_020020_genvalenciana_interurbano',
  '20260901_200004_emt_valencia',
  '20260902_080005_metro_valencia',
  '20260702_060101_ouigo',
  '20260908_000005_renfe_avld'],
 'files': ['valencia.gpkg::study_area_boundary',
  'valencia.gpkg::osm_driving_nodes',
  'valencia.gpkg::osm_driving_edges',
  'valencia.gpkg::osm_bike_nodes',
  'valencia.gpkg::osm_bike_edges',
  'valencia.gpkg::osm_walking_nodes',
  'valencia.gpkg::osm_walking_edges',
  'valencia.gpkg::gtfs_20260908_000019_bus_alsa_nodes',
  'valencia.gpkg::gtfs_20260908_000019_bus_alsa_edges',
  'valencia.gpkg::gtfs_20260908_020020_genvalenciana_interurbano_nodes',
  'valencia.gpkg::gtfs_20260908_020020_genvalen

In [18]:
#Driving
df_driving_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "osm_driving_edges")
df_driving_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "osm_driving_nodes")

#Walking
df_walking_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "osm_walking_nodes")
df_walking_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "osm_walking_edges")

#Bike
df_bike_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "osm_bike_nodes")
df_bike_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "osm_bike_edges")

#Train
df_train_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "gtfs_20260902_080005_metro_valencia_nodes")
df_train_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "gtfs_20260902_080005_metro_valencia_edges")

#Bus
df_bus_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "gtfs_20260908_020020_genvalenciana_interurbano_nodes")
df_bus_nodes = gpd.read_file(f"{output_folder}/valencia.gpkg", layer= "gtfs_20260908_020020_genvalenciana_interurbano_edges")


### Driving

In [19]:
run_all_checks_osm(df_driving_nodes, df_driving_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[FAIL (1)] edges: expected columns present
['from_node_id']

[INFO] edges: extra/unexpected columns: ['from_node_']

[OK] no fully duplicate rows
[OK] edge_id not null
[FAIL (77658)] from_node_id not null
    edge_id                     from_node_                     to_node_id      highway lanes maxspeed                   name  oneway  reversed     length       tts                                                     geometry
0   5011435                       25767293  driving_topology_node_0000002    secondary     5       50       Carrer de Xàtiva    True     False  13.335806  0.960178   LINESTRING (725813.199 4371905.836, 725799.924 4371904.26)
1   5011435  driv

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry not null', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name="node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']", level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='geometry parses as valid POINT', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edges: expected columns present', level='fail', n_issues=1, issues=['from_node_id'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='edge_id not null', level='fail', n_issues=0, issues=None, note='

### Walking

In [20]:
run_all_checks_osm(df_walking_nodes, df_walking_edges)

=== NODES ===
--- nodes: schema ---
[FAIL (2)] nodes: expected columns present
['node_id', 'node_role']

[INFO] nodes: extra/unexpected columns: ['edge_id', 'from_node_id', 'highway', 'lanes', 'length', 'maxspeed', 'name', 'oneway', 'reversed', 'to_node_id', 'tts']

[OK] no fully duplicate rows
[OK] node_id is unique
[FAIL (387360)] node_id not null
     edge_id                   from_node_id                     to_node_id      highway lanes maxspeed                   name  oneway  reversed    length       tts                                                     geometry
0  803049226                       25767290                    11420770018      footway  None     None                   None   False     False  8.794706  6.332188  LINESTRING (725727.356 4372260.811, 725723.068 4372268.485)
1  803049226                       25767290                    11420770019      footway  None     None                   None   False      True  4.274938  3.077955  LINESTRING (725727.356 4372260.81

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=2, issues=['node_id', 'node_role'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=387360, issues=           edge_id                   from_node_id  \
 0        803049226                       25767290   
 1        803049226                       25767290   
 2        803466250                       25767290   
 3        803466250                       25767290   
 4        803466250  walking_topology_node_0000005   
 ...            ...                            ...   
 387355  1545361355  walking_topology_node_0124410   
 387356  1545361355  walking_topology_node_0124409   
 387357  1545361355  walking_topology_node_0124408   
 387358  1545361355  walking_topology_node_0124407   
 387359  1545361359        

### Bike

In [21]:
run_all_checks_osm(df_bike_nodes, df_bike_edges)

=== NODES ===
--- nodes: schema ---
[FAIL (2)] nodes: expected columns present
['node_id', 'node_role']

[INFO] nodes: extra/unexpected columns: ['edge_id', 'from_node_id', 'highway', 'lanes', 'length', 'maxspeed', 'name', 'oneway', 'reversed', 'to_node_id', 'tts']

[OK] no fully duplicate rows
[OK] node_id is unique
[FAIL (201023)] node_id not null
     edge_id from_node_id                  to_node_id     highway lanes maxspeed                               name  oneway  reversed      length        tts                                                     geometry
0    5098789     25767292                  6766690060  pedestrian  None     None         Carrer de Martínez Cubells   False     False   56.652682  16.596644  LINESTRING (725789.942 4372033.731, 725846.733 4372032.464)
1  583027084     25767292                    35594380  pedestrian     1     None                 Passeig de Russafa   False     False   32.524900  10.805976  LINESTRING (725789.942 4372033.731, 725782.554 4372065

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=2, issues=['node_id', 'node_role'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=201023, issues=           edge_id                from_node_id                  to_node_id  \
 0          5098789                    25767292                  6766690060   
 1        583027084                    25767292                    35594380   
 2        583027084                    25767292                  5574601881   
 3        583027108                    25767292                    25767445   
 4          5011435                    25767293  bike_topology_node_0000007   
 ...            ...                         ...                         ...   
 201018  1545361355  bike_topology_node_0085935  bike_topology_node_

### Bus

In [22]:
run_all_checks_osm(df_bus_nodes, df_bus_edges)

=== NODES ===
--- nodes: schema ---
[FAIL (2)] nodes: expected columns present
['node_id', 'node_role']

[INFO] nodes: extra/unexpected columns: ['days_of_week_summary', 'edge_id', 'from_node_id', 'hourly_travel_times', 'hourly_trip_counts', 'route_id', 'route_long_name', 'route_short_name', 'to_node_id', 'trip_count', 'trip_count_peak_am', 'trip_count_peak_am_weekday', 'trip_count_peak_am_weekend', 'trip_count_peak_pm', 'trip_count_peak_pm_weekday', 'trip_count_peak_pm_weekend', 'trip_count_rest_of_day', 'trip_count_rest_of_day_weekday', 'trip_count_rest_of_day_weekend', 'trip_count_weekday', 'trip_count_weekend', 'tts_max', 'tts_mean', 'tts_mean_peak_am', 'tts_mean_peak_am_weekday', 'tts_mean_peak_am_weekend', 'tts_mean_peak_pm', 'tts_mean_peak_pm_weekday', 'tts_mean_peak_pm_weekend', 'tts_mean_rest_of_day', 'tts_mean_rest_of_day_weekday', 'tts_mean_rest_of_day_weekend', 'tts_mean_weekday', 'tts_mean_weekend', 'tts_min']

[OK] no fully duplicate rows
[OK] node_id is unique
[FAIL (219

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=2, issues=['node_id', 'node_role'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=219, issues=                                               edge_id  from_node_id  \
 0    20260908_020020_genvalenciana_interurbano_1370...     137000122   
 1    20260908_020020_genvalenciana_interurbano_1410...     141001105   
 2    20260908_020020_genvalenciana_interurbano_1410...     141001105   
 3    20260908_020020_genvalenciana_interurbano_1410...     141001105   
 4    20260908_020020_genvalenciana_interurbano_1410...     141001105   
 ..                                                 ...           ...   
 214  20260908_020020_genvalenciana_interurbano_9259...  925918510213   
 215  20260908_020020_genvalenciana_inte

### Train

In [23]:
run_all_checks_osm(df_train_nodes, df_train_edges)

=== NODES ===
--- nodes: schema ---
[FAIL (2)] nodes: expected columns present
['node_id', 'node_role']

[INFO] nodes: extra/unexpected columns: ['days_of_week_summary', 'edge_id', 'from_node_id', 'hourly_travel_times', 'hourly_trip_counts', 'route_id', 'route_long_name', 'route_short_name', 'to_node_id', 'trip_count', 'trip_count_peak_am', 'trip_count_peak_am_weekday', 'trip_count_peak_am_weekend', 'trip_count_peak_pm', 'trip_count_peak_pm_weekday', 'trip_count_peak_pm_weekend', 'trip_count_rest_of_day', 'trip_count_rest_of_day_weekday', 'trip_count_rest_of_day_weekend', 'trip_count_weekday', 'trip_count_weekend', 'tts_max', 'tts_mean', 'tts_mean_peak_am', 'tts_mean_peak_am_weekday', 'tts_mean_peak_am_weekend', 'tts_mean_peak_pm', 'tts_mean_peak_pm_weekday', 'tts_mean_peak_pm_weekend', 'tts_mean_rest_of_day', 'tts_mean_rest_of_day_weekday', 'tts_mean_rest_of_day_weekend', 'tts_mean_weekday', 'tts_mean_weekend', 'tts_min']

[OK] no fully duplicate rows
[OK] node_id is unique
[FAIL (162

[CheckResult(name='nodes: expected columns present', level='fail', n_issues=2, issues=['node_id', 'node_role'], note=''),
 CheckResult(name='no fully duplicate rows', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id is unique', level='fail', n_issues=0, issues=None, note=''),
 CheckResult(name='node_id not null', level='fail', n_issues=1621, issues=                                     edge_id from_node_id to_node_id  \
 0     20260902_080005_metro_valencia_100_102          100        102   
 1     20260902_080005_metro_valencia_102_100          102        100   
 2     20260902_080005_metro_valencia_102_100          102        100   
 3      20260902_080005_metro_valencia_108_23          108         23   
 4      20260902_080005_metro_valencia_108_23          108         23   
 ...                                      ...          ...        ...   
 1616    20260902_080005_metro_valencia_99_95           99         95   
 1617    20260902_080005_metro_valencia